# 04 — Grafos Dinámicos

**Fase 4 del proyecto ISO 20022 en Redes DLT.**

Toma el dataset etiquetado de `data/processed/txs_etiquetadas.csv` y modela la red de transacciones como grafos dirigidos por ventana temporal (emisor → receptor). Por cada ventana produce un HTML interactivo (pyvis) y acumula un CSV con métricas de red (grado, centralidad, densidad). Los resultados se guardan en `outputs/grafos/`.

> **Nota:** en v1 Stellar no aporta destino a nivel transacción, por lo que sus filas quedan fuera del grafo hasta enriquecer la extracción con `/operations`. XRPL y Hedera sí tienen emisor y receptor.

# 1. Configuración

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = False  # activa guardado de HTML y CSV

# 2. Imports

In [ ]:
import os

import pandas as pd
import networkx as nx

# pyvis se importa de forma diferida dentro de exportar_html
# para no exigirlo cuando SAVE_DATA=False

# 3. Paths y Constantes

In [ ]:
# Paths de entrada / salida
BASE_DIR: str = "tesis"
ARCHIVO_ENTRADA: str = os.path.join(
    BASE_DIR, "data", "processed", "txs_etiquetadas.csv")
DIR_GRAFOS: str = os.path.join(
    BASE_DIR, "outputs", "grafos")

# Constantes de análisis
VENTANA: str = "7D"          # tamaño de la ventana temporal
MAX_NODOS_VIZ: int = 150     # nodos máx. por HTML (pyvis)
CAP_BETWEENNESS: int = 500   # tope de nodos para betweenness
COLUMNAS: list[str] = [
    "tx_id", "red", "timestamp", "emisor", "receptor",
    "amount", "currency", "fee", "memo_raw",
    "memo_decoded", "iso_code", "iso_metodo",
]

# 4. Funciones

### 4.a — Carga del dataset

In [ ]:
def cargar_dataset(ruta: str) -> pd.DataFrame:
    """Carga el dataset etiquetado y parsea el timestamp."""
    if not os.path.exists(ruta):
        if DEBUG:
            print(f"[DEBUG] no existe {ruta}")
        return pd.DataFrame(columns=COLUMNAS)
    df = pd.read_csv(ruta)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["timestamp"], utc=True, errors="coerce")
    if DEBUG:
        print(f"[DEBUG] filas cargadas: {len(df)}")
    return df

### 4.b — Ventanas temporales y construcción del grafo

In [ ]:
def generar_ventanas(df: pd.DataFrame,
                     ventana: str) -> list[tuple]:
    """
    Divide el dataset en sub-DataFrames por ventana temporal.

    Solo conserva filas con emisor y receptor válidos.

    Returns:
        Lista de tuplas (etiqueta_fecha, sub_dataframe).
    """
    base = df.dropna(subset=["timestamp", "emisor",
                             "receptor"]).copy()
    base = base[(base["emisor"] != "")
                & (base["receptor"] != "")]
    if base.empty:
        return []
    grupos = base.groupby(
        pd.Grouper(key="timestamp", freq=ventana))
    ventanas: list[tuple] = []
    for etiqueta, sub in grupos:
        if sub.empty:
            continue
        ventanas.append((etiqueta.date().isoformat(), sub))
    return ventanas


def construir_grafo(sub: pd.DataFrame) -> nx.DiGraph:
    """
    Construye un grafo dirigido emisor -> receptor.

    Cada arista pondera el nº de transacciones (weight) y
    acumula el monto total (monto).
    """
    grafo = nx.DiGraph()
    agrupado = (
        sub.groupby(["emisor", "receptor"])
        .agg(
            peso=("tx_id", "count"),
            monto=("amount", "sum"),
        )
        .reset_index()
    )
    for fila in agrupado.itertuples(index=False):
        monto = (float(fila.monto)
                 if pd.notna(fila.monto) else 0.0)
        grafo.add_edge(
            fila.emisor, fila.receptor,
            weight=int(fila.peso), monto=monto)
    return grafo

### 4.c — Métricas de red

In [ ]:
def metricas_grafo(grafo: nx.DiGraph,
                   etiqueta: str) -> dict:
    """
    Calcula métricas de red para una ventana: tamaño,
    densidad, grado y centralidad.
    """
    n = grafo.number_of_nodes()
    if n == 0:
        return {
            "ventana": etiqueta, "n_nodos": 0,
            "n_aristas": 0, "densidad": 0.0,
            "grado_medio": 0.0, "grado_max": 0,
            "nodo_grado_max": "", "centralidad_grado_max": 0.0,
            "betweenness_max": 0.0,
        }
    grados = dict(grafo.degree())
    nodo_max = max(grados, key=grados.get)
    deg_cent = nx.degree_centrality(grafo)
    if n <= CAP_BETWEENNESS:
        btw = nx.betweenness_centrality(grafo)
        btw_max = max(btw.values()) if btw else 0.0
    else:
        btw_max = float("nan")
    return {
        "ventana": etiqueta,
        "n_nodos": n,
        "n_aristas": grafo.number_of_edges(),
        "densidad": nx.density(grafo),
        "grado_medio": sum(grados.values()) / n,
        "grado_max": grados[nodo_max],
        "nodo_grado_max": nodo_max,
        "centralidad_grado_max": max(deg_cent.values()),
        "betweenness_max": btw_max,
    }

### 4.d — Exportación interactiva (pyvis)

In [ ]:
def podar_grafo(grafo: nx.DiGraph,
                top_n: int) -> nx.DiGraph:
    """Conserva los top_n nodos de mayor grado (solo viz)."""
    if grafo.number_of_nodes() <= top_n:
        return grafo
    grados = dict(grafo.degree())
    top = sorted(grados, key=grados.get,
                 reverse=True)[:top_n]
    return grafo.subgraph(top).copy()


def exportar_html(grafo: nx.DiGraph, nombre: str) -> None:
    """Exporta el grafo como HTML interactivo si procede."""
    if not SAVE_DATA or grafo.number_of_nodes() == 0:
        return
    from pyvis.network import Network
    os.makedirs(DIR_GRAFOS, exist_ok=True)
    sub = podar_grafo(grafo, MAX_NODOS_VIZ)
    net = Network(
        height="700px", width="100%",
        directed=True, notebook=False)
    net.from_nx(sub)
    ruta = os.path.join(DIR_GRAFOS, nombre)
    net.save_graph(ruta)
    if DEBUG:
        print(f"[DEBUG] grafo: {ruta}")

# 5. Proceso

In [ ]:
df = cargar_dataset(ARCHIVO_ENTRADA)
ventanas = generar_ventanas(df, VENTANA)
if DEBUG:
    print(f"[DEBUG] ventanas generadas: {len(ventanas)}")

# Un grafo y sus métricas por cada ventana temporal
filas_metricas: list[dict] = []
for etiqueta, sub in ventanas:
    grafo = construir_grafo(sub)
    filas_metricas.append(metricas_grafo(grafo, etiqueta))
    exportar_html(grafo, f"grafo_{etiqueta}.html")

metricas = pd.DataFrame(filas_metricas)
if DEBUG and not metricas.empty:
    print(metricas.head())

# Guardado del CSV de métricas de red
if SAVE_DATA and not metricas.empty:
    os.makedirs(DIR_GRAFOS, exist_ok=True)
    ruta = os.path.join(DIR_GRAFOS, "metricas_red.csv")
    metricas.to_csv(ruta, index=False)
    if DEBUG:
        print(f"[DEBUG] guardado: {ruta}")